# SEACO-6161 · 03 Weekly VACUUM job · `dbw-arc-merone-pipeline-prod`

This stops the storage from growing back (Phase 3 of the ticket). It is meant to run as a **scheduled
Databricks job**, weekly. Each run:

1. finds the Delta tables on the root account written in the last `active_days` days
2. skips shallow-clone sources (from the latest `01_investigation` run) and anything in `exclude_tables`
3. runs `VACUUM … RETAIN 168 HOURS` on each, and records before and after sizes in `cleanup_log`
4. **fails the run** (so the failure email goes out) when a table's on-disk size is still more than
   `alert_ratio` × its live size

Run it by hand with `dry_run = true` first. The job definition sets `dry_run = false`.
Run `01_investigation` again about once a month, so the clone-source list stays current.

In [0]:
%run ./00_config_and_helpers

In [0]:
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "1. Dry run")
dbutils.widgets.text("retain_hours", "168", "2. Retain hours (>= 168)")
dbutils.widgets.text("active_days", "14", "3. Tables written in the last N days")
dbutils.widgets.text("alert_ratio", "2.0", "4. Alert when disk / live > ratio")
dbutils.widgets.text("alert_min_gib", "20", "5. ...and disk >= GiB")
dbutils.widgets.dropdown("fail_on_alert", "true", ["true", "false"], "6. Fail the run on alert")
dbutils.widgets.text("exclude_tables", "", "7. Never VACUUM (db.table, comma list)")

DRY_RUN = dbutils.widgets.get("dry_run") != "false"
RETAIN_HOURS = int(dbutils.widgets.get("retain_hours"))
ACTIVE_DAYS = int(dbutils.widgets.get("active_days"))
ALERT_RATIO = float(dbutils.widgets.get("alert_ratio"))
ALERT_MIN = float(dbutils.widgets.get("alert_min_gib")) * GIB
FAIL_ON_ALERT = dbutils.widgets.get("fail_on_alert") == "true"
EXCLUDE = {t.strip() for t in dbutils.widgets.get("exclude_tables").split(",") if t.strip()}
assert RETAIN_HOURS >= MIN_RETAIN_HOURS, f"retain_hours must be >= {MIN_RETAIN_HOURS}"

In [0]:
# Clone sources from the latest investigation. With no investigation, refuse to run: VACUUM without a
# clone check is how shallow-clone backups get broken.
inv, inv_ts = latest("table_inventory")
CLONE_SOURCE_NAMES = {r["full_name"] for r in inv.filter("is_shallow_clone_source").select("full_name").collect()}
inv_age = (now_utc() - inv_ts).days
print(f"Clone sources protected: {len(CLONE_SOURCE_NAMES)} (from inventory {inv_age} days old)")
if inv_age > 45:
    print("⚠️  Inventory older than 45 days; new shallow clones may be missed. Re-run 01_investigation.")

cutoff = now_utc() - dt.timedelta(days=ACTIVE_DAYS)


def check(db, t):
    try:
        ext = table_extended(db, t)
        if (ext.get("Provider") or "").lower() != "delta":
            return None
        d = describe_detail(db, t)
        if classify_path(d["location"]) != "root" or (d.get("lastModified") or dt.datetime.min) < cutoff:
            return None
        return dict(full_name=f"{db}.{t}", location=d["location"], live=d.get("sizeInBytes") or 0)
    except Exception:
        return None


todo = [(db, t) for db in list_databases() if db != AUDIT_DB for t in list_tables(db)]
with ThreadPoolExecutor(THREADS) as pool:
    active = [r for r in pool.map(lambda x: check(*x), todo) if r]
targets = [r for r in active if r["full_name"] not in CLONE_SOURCE_NAMES | EXCLUDE]
print(f"{len(todo):,} tables scanned, {len(active)} active on root, {len(targets)} to VACUUM "
      f"({len(active) - len(targets)} skipped as clone source / excluded)")

In [0]:
spark.conf.set("spark.databricks.delta.vacuum.parallelDelete.enabled", "true")
alerts, freed_total = [], 0
for r in sorted(targets, key=lambda x: -x["live"]):
    db, t = r["full_name"].split(".", 1)
    t0 = time.time()
    try:
        cs_before = content_summary(r["location"])
        before, fb = cs_before["bytes"] or 0, cs_before["files"] or 0
        if DRY_RUN:
            n = spark.sql(f"VACUUM {qname(db, t)} RETAIN {RETAIN_HOURS} HOURS DRY RUN").count()
            after, fa = before, fb
            print(f"  DRY {r['full_name']:<60} {human(before):>12} on disk, {fb:,} files, {n:,} files listed")
        else:
            spark.sql(f"VACUUM {qname(db, t)} RETAIN {RETAIN_HOURS} HOURS")
            cs_after = content_summary(r["location"])
            after, fa = cs_after["bytes"] or 0, cs_after["files"] or 0
            freed_total += before - after
            print(f"  {r['full_name']:<60} files {fb:>12,} -> {fa:>12,}  "
                  f"freed {human(before - after):>12} in {time.time() - t0:,.0f}s")
        log_action("weekly_vacuum", r["full_name"], DRY_RUN, "ok", before, after, time.time() - t0,
                   files_before=fb, files_after=fa)
        if after >= ALERT_MIN and r["live"] and after / r["live"] > ALERT_RATIO:
            alerts.append((r["full_name"], after, r["live"]))
    except Exception as e:
        print(f"  ! {r['full_name']}: {str(e)[:300]}")
        log_action("weekly_vacuum", r["full_name"], DRY_RUN, "error", detail=str(e))
print(f"\n{'Dry run' if DRY_RUN else 'Freed'}: {human(freed_total)}")

In [0]:
if alerts:
    msg = "\n".join(f"  {n}: {human(d)} on disk vs {human(l)} live ({d / l:.1f}x)" for n, d, l in alerts)
    print(f"⚠️  {len(alerts)} tables still above {ALERT_RATIO}x live size after VACUUM. These are usually "
          f"shallow-clone sources, or tables with many small rewrites inside the 7-day window:\n{msg}")
    if FAIL_ON_ALERT and not DRY_RUN:
        raise Exception(f"SEACO-6161 storage alert: {len(alerts)} tables above {ALERT_RATIO}x live size")
else:
    print("No storage alerts.")

## Job definition

Create the job in the Workflows UI, or with `POST /api/2.1/jobs/create` and the JSON below.
Replace the `<...>` placeholders first: the notebook folder, an LTS runtime and the team distribution list.
The cluster must use `SINGLE_USER` access mode.

```json
{
  "name": "SEACO-6161-weekly-dbfs-vacuum (dbw-arc-merone-pipeline-prod)",
  "schedule": {
    "quartz_cron_expression": "0 0 3 ? * SUN",
    "timezone_id": "Europe/Amsterdam",
    "pause_status": "UNPAUSED"
  },
  "max_concurrent_runs": 1,
  "email_notifications": {
    "on_failure": [
      "<team-distribution-list>"
    ]
  },
  "tasks": [
    {
      "task_key": "weekly_vacuum",
      "notebook_task": {
        "notebook_path": "<folder>/03_weekly_vacuum_job",
        "base_parameters": {
          "dry_run": "false",
          "fail_on_alert": "true"
        }
      },
      "job_cluster_key": "vacuum"
    }
  ],
  "job_clusters": [
    {
      "job_cluster_key": "vacuum",
      "new_cluster": {
        "spark_version": "<LTS runtime>",
        "node_type_id": "Standard_D8ds_v5",
        "num_workers": 4,
        "data_security_mode": "SINGLE_USER"
      }
    }
  ],
  "tags": {
    "ticket": "SEACO-6161",
    "team": "jediknights"
  }
}
```